# 04 Load Schema v0.1 into Neo4j Aura

**Date:** 2026-10-15
**Author:** Xincheng (George) Ding
**Branch:** `george`
**Needs network** (Aura Free). Fill `NEO4J_*` in `fa26/.env` first.

**Purpose:** Load the prototype CSVs into Aura using schema v0.1 from notebook 02 (decoded code labels, PROVIDED_BY / RECEIVED_BY edges, constraints and indexes). Idempotent: uses MERGE, safe to re-run. Required before notebook 03 can run with `DRY_RUN = False`.

In [ ]:
from pathlib import Path
import os, pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

NB_DIR = Path.cwd(); FA26 = NB_DIR.parent
GRAPH_DIR = FA26.parent.parent / "reference" / "iati-graph-prototype" / "data" / "graph"
load_dotenv(FA26 / ".env")
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity(); print("Neo4j connected")
DB = os.environ.get("NEO4J_DATABASE", "neo4j")
tables = {p.stem: pd.read_csv(p, dtype=str, keep_default_na=False) for p in sorted(GRAPH_DIR.glob("*.csv"))}

In [ ]:
TRANSACTION_TYPE = {"1": "Incoming Funds", "2": "Outgoing Commitment", "3": "Disbursement", "4": "Expenditure",
                    "11": "Incoming Commitment", "12": "Outgoing Pledge", "13": "Incoming Pledge"}
ORG_TYPE = {"10": "Government", "11": "Local Government", "15": "Other Public Sector", "21": "International NGO",
            "22": "National NGO", "23": "Regional NGO", "24": "Partner Country based NGO", "40": "Multilateral",
            "60": "Foundation", "70": "Private Sector", "80": "Academic, Training and Research", "90": "Other"}
SECTOR_VOCAB = {"1": "OECD DAC 5-digit", "2": "OECD DAC 3-digit", "7": "SDG Goal", "8": "SDG Target",
                "10": "Humanitarian Global Clusters", "99": "Reporting Organisation"}

def prep(csv):
    df = tables[csv].copy()
    if csv == "nodes_transactions":
        df["type_label"] = df["type"].map(TRANSACTION_TYPE).fillna("Unknown")
        df["value_num"] = pd.to_numeric(df["value"], errors="coerce")
    if csv == "nodes_organisations": df["type_label"] = df["type"].map(ORG_TYPE).fillna("Unknown")
    if csv == "nodes_sectors":       df["vocabulary_label"] = df["vocabulary"].map(SECTOR_VOCAB).fillna("Unknown")
    if csv == "nodes_activities":    df["hunger_basis"] = df["match_basis"]
    return df.astype(object).where(df.notna(), None)

NODES = {"nodes_activities": ("Activity", "iati_identifier"), "nodes_organisations": ("Organisation", "ref"),
         "nodes_countries": ("Country", "code"), "nodes_sectors": ("Sector", "sector_key"),
         "nodes_locations": ("Location", "location_id"), "nodes_transactions": ("Transaction", "tx_id"),
         "nodes_budgets": ("Budget", "budget_id"), "nodes_sdgs": ("SDG", "sdg_key")}
RELS = [("rel_reports", "REPORTS", "Organisation", "org_ref", "ref", "Activity", "iati_identifier", "iati_identifier", None),
        ("rel_participates", "PARTICIPATES_IN", "Organisation", "org_ref", "ref", "Activity", "iati_identifier", "iati_identifier", None),
        ("rel_targets_country", "TARGETS", "Activity", "iati_identifier", "iati_identifier", "Country", "country_code", "code", None),
        ("rel_classified_as", "CLASSIFIED_AS", "Activity", "iati_identifier", "iati_identifier", "Sector", "sector_key", "sector_key", None),
        ("rel_located_at", "LOCATED_AT", "Activity", "iati_identifier", "iati_identifier", "Location", "location_id", "location_id", None),
        ("rel_contributes_to", "CONTRIBUTES_TO", "Activity", "iati_identifier", "iati_identifier", "SDG", "sdg_key", "sdg_key", None),
        ("nodes_transactions", "HAS_TRANSACTION", "Activity", "iati_identifier", "iati_identifier", "Transaction", "tx_id", "tx_id", None),
        ("nodes_budgets", "HAS_BUDGET", "Activity", "iati_identifier", "iati_identifier", "Budget", "budget_id", "budget_id", None),
        ("rel_transaction_edges", "PROVIDED_BY", "Transaction", "tx_id", "tx_id", "Organisation", "org_ref", "ref", ("edge", "PROVIDED_BY")),
        ("rel_transaction_edges", "RECEIVED_BY", "Transaction", "tx_id", "tx_id", "Organisation", "org_ref", "ref", ("edge", "RECEIVED_BY"))]

In [ ]:
DDL = [
 "CREATE CONSTRAINT activity_id  IF NOT EXISTS FOR (n:Activity)     REQUIRE n.iati_identifier IS UNIQUE",
 "CREATE CONSTRAINT org_ref      IF NOT EXISTS FOR (n:Organisation) REQUIRE n.ref IS UNIQUE",
 "CREATE CONSTRAINT country_code IF NOT EXISTS FOR (n:Country)      REQUIRE n.code IS UNIQUE",
 "CREATE CONSTRAINT sector_key   IF NOT EXISTS FOR (n:Sector)       REQUIRE n.sector_key IS UNIQUE",
 "CREATE CONSTRAINT location_id  IF NOT EXISTS FOR (n:Location)     REQUIRE n.location_id IS UNIQUE",
 "CREATE CONSTRAINT tx_id        IF NOT EXISTS FOR (n:Transaction)  REQUIRE n.tx_id IS UNIQUE",
 "CREATE CONSTRAINT budget_id    IF NOT EXISTS FOR (n:Budget)       REQUIRE n.budget_id IS UNIQUE",
 "CREATE CONSTRAINT sdg_key      IF NOT EXISTS FOR (n:SDG)          REQUIRE n.sdg_key IS UNIQUE",
 "CREATE INDEX activity_publisher IF NOT EXISTS FOR (n:Activity)     ON (n.publisher)",
 "CREATE INDEX activity_hunger    IF NOT EXISTS FOR (n:Activity)     ON (n.hunger_basis)",
 "CREATE INDEX org_name           IF NOT EXISTS FOR (n:Organisation) ON (n.name)",
 "CREATE INDEX tx_type            IF NOT EXISTS FOR (n:Transaction)  ON (n.type_label)",
 "CREATE INDEX tx_date            IF NOT EXISTS FOR (n:Transaction)  ON (n.date)",
 "CREATE INDEX country_tier       IF NOT EXISTS FOR (n:Country)      ON (n.hotspot_tier)",
]
with driver.session(database=DB) as s:
    for q in DDL: s.run(q)
print("constraints and indexes ready")

In [ ]:
def batched(recs, size=2000):
    for i in range(0, len(recs), size): yield recs[i:i + size]

with driver.session(database=DB) as s:
    for csv, (label, key) in NODES.items():
        recs = prep(csv).to_dict("records")
        for b in batched(recs):
            s.run(f"UNWIND $rows AS r MERGE (n:{label} {{{key}: r.{key}}}) SET n += r", rows=b)
        print(f"{label:<14} {len(recs):>7} nodes")
    for csv, rtype, fl, fc, fk, tl, tc, tk, flt in RELS:
        df = tables[csv]
        if flt: df = df[df[flt[0]] == flt[1]]
        props = [c for c in df.columns if c not in (fc, tc, "edge")] if csv.startswith("rel_") else []
        rows = [{"f": r[fc], "t": r[tc], "p": {k: r[k] for k in props}} for r in df.to_dict("records")]
        q = (f"UNWIND $rows AS r MATCH (a:{fl} {{{fk}: r.f}}) MATCH (b:{tl} {{{tk}: r.t}}) "
             f"MERGE (a)-[x:{rtype}]->(b) SET x += r.p")
        for b in batched(rows): s.run(q, rows=b)
        print(f"{rtype:<16} {len(rows):>7} relationships")

In [ ]:
with driver.session(database=DB) as s:
    print(pd.DataFrame(s.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS n ORDER BY n DESC").data()))
    print(pd.DataFrame(s.run("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS n ORDER BY n DESC").data()))
    print("isolated:", s.run("MATCH (n) WHERE NOT (n)--() RETURN count(n) AS c").single()["c"])
driver.close()